In [5]:
# !pip --version

In [4]:
# %pip install langchain>=1.0.0 langchain-openai langchain-anthropic langchain-google-genai langchain-text-splitters tiktoken python-dotenv langchain-community faiss-cpu

In [8]:
import os

import langchain
import langchain_core
import langchain_openai

from dotenv import load_dotenv


# .env 로드
load_dotenv()


# API Key 확인
print(
    "[OK] OpenAI API 키 설정됨"
    if os.getenv("OPENAI_API_KEY")
    else "[ERROR] OpenAI API 키 미설정"
)

# LangChain Key
print(
    "[OK] LangChain 키 설정됨"
    if os.getenv("LANGSMITH_API_KEY")
    else "[ERROR] LangChain 키 미설정"
)

# LangChain 버전 확인
print(f"LangChain: {langchain.__version__}")
print(f"LangChain Core: {langchain_core.__version__}")
print(f"LangChain OpenAI: {langchain_openai.__version__}")

[OK] OpenAI API 키 설정됨
[OK] LangChain 키 설정됨
LangChain: 1.4.0
LangChain Core: 1.6.3
LangChain OpenAI: 1.6.2


In [2]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-4o-mini", temperature=0.2, max_tokens=2048)

In [1]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.prompts import PromptTemplate

In [10]:
email_conversation = """
From: 김철수 (chulsoo.kim@bikecorporation.me)
To: 이은채 (eunchae@teddyinternational.me)
Subject: "ZENESIS" 자전거 유통 협력 및 미팅 일정 제안

안녕하세요, 이은채 대리님,

저는 바이크코퍼레이션의 김철수 상무입니다. 최근 보도자료를 통해 귀사의 신규 자전거 "ZENESIS"에 대해 알게 되었습니다. 바이크코퍼레이션은 자전거 제조 및 유통 분야에서 혁신과 품질을 선도하는 기업으로, 이 분야에서의 장기적인 경험과 전문성을 가지고 있습니다.

ZENESIS 모델에 대한 상세한 브로슈어를 요청드립니다. 특히 기술 사양, 배터리 성능, 그리고 디자인 측면에 대한 정보가 필요합니다. 이를 통해 저희가 제안할 유통 전략과 마케팅 계획을 보다 구체화할 수 있을 것입니다.

또한, 협력 가능성을 더 깊이 논의하기 위해 다음 주 화요일(1월 15일) 오전 10시에 미팅을 제안합니다. 귀사 사무실에서 만나 이야기를 나눌 수 있을까요?

감사합니다.

김철수
상무이사
바이크코퍼레이션
"""

In [ ]:
prompt = PromptTemplate.from_template(
    "다음의 이메일 내용중 중요한 내용을 추출해 주세요.\n\n{email_conversation}"
)

chain = prompt | llm

answer = chain.stream({"email_conversation": email_conversation})

for chunk in answer:
    print(chunk.content, end="", flush=True)

중요 내용 요약:

- 발신자: 김철수 (바이크코퍼레이션 상무)
- 수신자: 이은채 (Teddy International)
- 주제: "ZENESIS" 자전거 유통 협력 및 미팅 일정 제안
- 요청 사항: ZENESIS 모델에 대한 상세 브로슈어 (기술 사양, 배터리 성능, 디자인 정보)
- 미팅 제안: 다음 주 화요일(1월 15일) 오전 10시, 귀사 사무실에서 만남 제안
- 목적: 협력 가능성 논의 및 유통 전략, 마케팅 계획 구체화

In [ ]:
class EmailSummary(BaseModel):
    person  : str = Field(description = "메일을 보낸 사람")
    email   : str = Field(description = "메일을 보낸 사람의 이메일 주소")
    subject : str = Field(description = "메일 제목")
    summary : str = Field(description = "메일 본문을 요약한 텍스트")
    date    : str = Field(description = "메일 본문에 언급된 미팅 날짜와 시간")
    
parser = PydanticOutputParser(pydantic_object=EmailSummary)

print(parser.get_format_instructions()) 

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"person": {"description": "메일을 보낸 사람", "title": "Person", "type": "string"}, "email": {"description": "메일을 보낸 사람의 이메일 주소", "title": "Email", "type": "string"}, "subject": {"description": "메일 제목", "title": "Subject", "type": "string"}, "summary": {"description": "메일 본문을 요약한 텍스트", "title": "Summary", "type": "string"}, "data": {"description": "메일 본문에 언급된 미팅 날짜와 시간", "title": "Data", "type": "string"}}, "required": ["person", "email", "subject", "summary", "data"]}
```


In [19]:
prompt = PromptTemplate.from_template(
    """
    You are helpful assistant. Please answer the follwing question in KOREA.
    
    QUESTION:
    {question}
    
    EMAIL CONVERSATION:
    {email_conversation}
    
    FORMAT:
    {format}
    """
)

prompt = prompt.partial(format = parser.get_format_instructions())

In [23]:
chain = prompt | llm | StrOutputParser()

In [32]:
response = chain.stream(
    {
        "email_conversation" : email_conversation,
        "question" : "이메일 내용중 주요 내용을 추출해 주세요.",
    }
)

for chunk in response:
    print(chunk, end="", flush=True)
    output += chunk

structured_output = parser.parse(output)
print(structured_output)

```json
{
  "person": "김철수",
  "email": "chulsoo.kim@bikecorporation.me",
  "subject": "\"ZENESIS\" 자전거 유통 협력 및 미팅 일정 제안",
  "summary": "김철수 상무가 이은채 대리님에게 'ZENESIS' 자전거에 대한 브로슈어 요청과 협력 논의를 위한 미팅 제안을 보냈습니다.",
  "data": "2024년 1월 15일 오전 10시"
}
```person='김철수' email='chulsoo.kim@bikecorporation.me' subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안' summary="김철수 상무가 이은채 대리에게 'ZENESIS' 자전거에 대한 브로슈어 요청과 협력 논의를 위한 미팅 제안을 함." data='2024년 1월 15일 오전 10시'


In [34]:
llm_with_structured = llm.with_structured_output(EmailSummary)

answer = llm_with_structured.invoke(email_conversation)

print(answer)


person='이은채' email='eunchae@teddyinternational.me' subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안' summary='김철수 상무가 이은채 대리에게 바이크코퍼레이션과 ZENESIS 자전거의 유통 협력에 대해 논의하기 위한 미팅을 제안하며, 상세한 브로슈어 요청과 미팅 일정을 제안하는 내용의 이메일입니다.' data='미팅 제안: 1월 15일 화요일 오전 10시, 귀사 사무실'
